# Transformación de datos de entrada para PyPOTS

In [5]:
import pandas as pd
import os

data_folder = os.environ.get("DATA_FOLDER_PATH_IPYNB", "../../data/")
file_processed = os.environ.get("PARQUET_PROCESSED_EDA", "df_500k_processed.parquet")
file_spatial = os.environ.get("PARQUET_SPATIAL_MERGE", "df_spatial_merge.parquet")
path_processed = os.path.join(data_folder, file_processed)
path_spatial = os.path.join(data_folder, file_spatial)

df = pd.read_parquet(path_processed)
df_spatial = pd.read_parquet(path_spatial)

In [6]:
# Match lan and lon to each DeviceId in the main dataset
coords_dict = df_spatial.set_index('DeviceId')[['lat', 'lon']].to_dict(orient='index')
df['lat'] = df['DeviceId'].map(lambda x: coords_dict.get(x, {}).get('lat'))
df['lon'] = df['DeviceId'].map(lambda x: coords_dict.get(x, {}).get('lon'))
df = df.dropna(subset=['lat', 'lon']) # remove rows without matching coordinates

print(f"Filas con coordenadas asignadas: {len(df)}")

# Ensure datetime format
for col in ['ArrivalTime', 'DepartureTime']:
    if not pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = pd.to_datetime(df[col])

# 4. Mostrar resumen final
display(df.head())

Filas con coordenadas asignadas: 260303


,DeviceId,ArrivalTime,DepartureTime,DurationMinutes,StreetMarker,SignPlateID,Sign,AreaName,StreetId,StreetName,...,SideOfStreetCode,SideName,BayId,InViolation,VehiclePresent,Hour,WeekDay,Month,lat,lon
0,23915,2019-11-03 03:00:56,2019-11-03 03:07:26,7.0,2374N,NaN,NaN,Windsor,123,BOURKE STREET,...,N,North,1753,False,False,3,6,11,-37.812649,144.968327
1,23914,2019-04-16 14:14:47,2019-04-16 14:15:23,1.0,1896N,287.0,1P MTR M-SAT 7:30-18:30,Twin Towers,528,COLLINS STREET,...,N,North,1532,False,False,14,1,4,-37.813710,144.972854
2,23913,2019-09-29 01:08:22,2019-09-29 01:31:41,23.0,1581S,NaN,NaN,Banks,670,FLINDERS STREET,...,S,South,1399,False,False,1,6,9,-37.818548,144.963329
3,23915,2019-06-05 18:59:17,2019-06-05 18:59:24,0.0,2374N,508.0,2P MTR M-SAT 18.30 - 20.30,Windsor,123,BOURKE STREET,...,N,North,1753,False,False,18,2,6,-37.812649,144.968327
4,23915,2019-04-01 07:03:41,2019-04-01 07:14:12,11.0,2374N,NaN,NaN,Windsor,123,BOURKE STREET,...,N,North,1753,False,True,7,0,4,-37.812649,144.968327


## Grid temporal

In [ ]:
import numpy as np
import pandas as pd

# Grid: each 15 minutes
grid_freq = os.environ.get("GRID_FREQ", "15min")
min_time = df['ArrivalTime'].min().floor(grid_freq)
max_time = df['DepartureTime'].max().ceil(grid_freq)
grid_time = pd.date_range(start=min_time, end=max_time, freq=grid_freq)

sensors_unique = df['DeviceId'].unique()
df_left_grid = pd.MultiIndex.from_product(
    [sensors_unique, grid_time],
    names=['DeviceId', 'Timestamp']
).to_frame(index=False)

left_df = df_left_grid.sort_values(['Timestamp', 'DeviceId']).reset_index(drop=True)
right_df = (
    df[['DeviceId', 'ArrivalTime', 'DepartureTime', 'VehiclePresent', 'lat', 'lon']]
    .sort_values(['ArrivalTime', 'DeviceId'])
    .reset_index(drop=True)
)

# Merge asof
df_grid = pd.merge_asof(
    left_df, right_df,
    left_on='Timestamp', right_on='ArrivalTime',
    by='DeviceId', direction='backward'
)

In [8]:
# Clean NAs where the car has already left
condition_na = df_grid['Timestamp'] > df_grid['DepartureTime']
df_grid['VehiclePresent'] = np.where(condition_na, np.nan, df_grid['VehiclePresent'])

# Fill lat/lon forward/backward
df_grid['lat'] = df_grid.groupby('DeviceId')['lat'].transform(lambda x: x.ffill().bfill())
df_grid['lon'] = df_grid.groupby('DeviceId')['lon'].transform(lambda x: x.ffill().bfill())

print(f"Dimensiones del grid temporal: {df_grid.shape}")

Dimensiones del grid temporal: (4239840, 7)


## Transformaciones seno y coseno

In [9]:
# Sin/cos tansformation
df_grid['Hour_sin'] = np.sin(2 * np.pi * df_grid['Timestamp'].dt.hour / 24.0)
df_grid['Hour_cos'] = np.cos(2 * np.pi * df_grid['Timestamp'].dt.hour / 24.0)
df_grid['WeekDay_sin'] = np.sin(2 * np.pi * df_grid['Timestamp'].dt.dayofweek / 7.0)
df_grid['WeekDay_cos'] = np.cos(2 * np.pi * df_grid['Timestamp'].dt.dayofweek / 7.0)
df_grid['Month_sin'] = np.sin(2 * np.pi * df_grid['Timestamp'].dt.month / 12.0)
df_grid['Month_cos'] = np.cos(2 * np.pi * df_grid['Timestamp'].dt.month / 12.0)

## Feature selection y exportación

In [10]:
features = [
    'DeviceId', 'Timestamp',  
    'VehiclePresent', 
    'lon', 'lat',
    'Hour_sin', 'Hour_cos', 
    'WeekDay_sin', 'WeekDay_cos', 
    'Month_sin', 'Month_cos'
]

df_model= df_grid[features].sort_values(by=['Timestamp', 'DeviceId']).reset_index(drop=True) # ensure sorted by DeviceId and Timestamp for time series modeling

print(f"Dimensiones del dataset para modelado: {df_model.shape}")
display(df_model.head())

Dimensiones del dataset para modelado: (4239840, 11)


,DeviceId,Timestamp,VehiclePresent,lon,lat,Hour_sin,Hour_cos,WeekDay_sin,WeekDay_cos,Month_sin,Month_cos
0,23913,2019-01-01,NaN,144.963329,-37.818548,0.0,1.0,0.781831,0.62349,0.5,0.866025
1,23914,2019-01-01,NaN,144.972854,-37.813710,0.0,1.0,0.781831,0.62349,0.5,0.866025
2,23915,2019-01-01,NaN,144.968327,-37.812649,0.0,1.0,0.781831,0.62349,0.5,0.866025
3,23916,2019-01-01,False,144.972239,-37.810486,0.0,1.0,0.781831,0.62349,0.5,0.866025
4,23917,2019-01-01,True,144.960474,-37.811671,0.0,1.0,0.781831,0.62349,0.5,0.866025


In [11]:
df_model_name = os.environ.get("PARQUET_MODEL", "df_pypots.parquet")
output_path = os.path.join(data_folder, df_model_name)
df_model.to_parquet(output_path)
print(f"Dataframe exportado a {output_path}")

Dataframe exportado a ../../data/df_pypots.parquet
